In [36]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as smp
from sympy.polys.polyfuncs import interpolate
from matplotlib.pyplot import figure
from scipy.interpolate import CubicSpline
from scipy.interpolate import PchipInterpolator
import ipywidgets as widgets
from ipywidgets import interact
from scipy.interpolate import UnivariateSpline
from IPython.display import display, Math
from scipy.special import roots_legendre
from decimal import Decimal, getcontext
import math
from functools import lru_cache
import sys

# Задача 1.

Рассчитать с машинной точностью интеграл с неопределенностью $\int_{-1}^{1} dx \sin(x/2) / (e^x - 1)$ квадратурой на разностных схемах.

In [7]:
###############################################################################
#  1) Вспомогательные функции для разностных квадратур                        #
###############################################################################
getcontext().prec = 100  # повышенная точность для Decimal

@lru_cache(None)
def factorial_dec(n: int) -> Decimal:
    if n < 0:
        raise ValueError("Negative factorial not defined")
    if n == 0 or n == 1:
        return Decimal(1)
    return Decimal(n) * factorial_dec(n - 1)

@lru_cache(None)
def comb_dec(n: int, k: int) -> Decimal:
    if k < 0 or k > n:
        return Decimal(0)
    return factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

@lru_cache(None)
def D(n: int, j: int) -> Decimal:
    """
    Вычисляет D_n^(j) по заданной рекуррентной формуле (см. теорию).
    """
    if j == 0:
        return Decimal(1)
    # "прогреваем" кэш:
    _ = D(n, j-1)
    return _compute_all_D_up_to(n, j)[j]

@lru_cache(None)
def _compute_all_D_up_to(n: int, J: int):
    results = [Decimal(0)]*(J+1)
    results[0] = Decimal(1)
    for j in range(J):
        s = Decimal(0)
        for k in range(n + 2*j + 1):
            for l in range(j+1):
                val = (
                    Decimal((-1)**k)
                    * results[l]
                    * comb_dec(n+2*l, k - (j - l))
                    * Decimal((n + 2*j - 2*k)**(n + 2*j + 2))
                    / (Decimal(2)**(n + 2*j + 2) * factorial_dec(n + 2*j + 2))
                )
                s += val
        # Финальная формула берёт только s, без + results[j].
        results[j+1] = s
    return results

@lru_cache(None)
def A(k: int, n: int, m: int) -> Decimal:
    """
    A_{k,n}^m = sum_{l=0}^m [ (-1)^(k-m) * D_n^(l) * C_{n+2l}^{k - m + l} ].
    """
    s = Decimal(0)
    sign = Decimal((-1)**(k - m))
    for l in range(m+1):
        s += (
            sign
            * D(n, l)
            * comb_dec(n+2*l, k - m + l)
        )
    return s

@lru_cache(None)
def W_list(m: int):
    """
    Коэффициенты W_k^m (k=0..2m) в Decimal.

      W_k^m = sum_{n=0}^m [ A_{k,2n}^{m-n} / (2^(2n)*(2n+1)!) ].
    """
    result = []
    for k in range(2*m + 1):
        s = Decimal(0)
        for n_ in range(m+1):
            a_val = A(k, 2*n_, m - n_)
            denom = (Decimal(2)**(2*n_) * factorial_dec(2*n_+1))
            s += a_val / denom
        result.append(s)
    return result

def build_function_values_diff_scheme(f, a: float, b: float, J: int, m: int):
    """
    Центральная сетка: x_j = a + (j+0.5)*h, j=-m..(J-1+m).
    Возвращает (y_values, h, calls).
    """
    h = (b - a)/J
    values = []
    for j_real in range(-m, (J - 1) + m + 1):
        x_val = a + (j_real + 0.5)*h
        values.append(f(x_val))
    calls = len(values)
    return values, h, calls

def integrate_by_diff_scheme(f, a: float, b: float, J: int, m: int):
    """
    Разностная (дифференсная) квадратура порядка 2m:
      ∫[a..b] f(x) dx ≈ h Σ_j Σ_k [ W_{m-k} * f( x_{j+k} ) ].
    Возвращает (approx, calls).
    """
    y_values, h, calls = build_function_values_diff_scheme(f, a, b, J, m)
    W_dec = W_list(m)
    W = [float(wd) for wd in W_dec]
    
    offset = m
    total_sum = 0.0
    for j in range(J):
        loc_sum = 0.0
        for k in range(-m, m+1):
            idx = (j + k) + offset
            loc_sum += W[m - k] * y_values[idx]
        total_sum += loc_sum
    return (h * total_sum, calls)

###############################################################################
#        2) Метод Симпсона (классический) с заданным числом вызовов f         #
###############################################################################

def build_function_values_simpson(f, a: float, b: float, n: int):
    h = (b - a)/n
    y = [f(a + i*h) for i in range(n+1)]
    return y, h

def simpson_by_calls(f, a: float, b: float, calls: int):
    """
    Метод Симпсона «под число вызовов calls».
    Если (calls-1) нечётно, уменьшаем n на 1.
    Если n < 2, берём n=2.
    Возвращает (approx, real_calls).
    """
    n = calls - 1
    if n % 2 == 1:
        n -= 1
    if n < 2:
        n = 2
    
    y, h = build_function_values_simpson(f, a, b, n)
    real_calls = n + 1
    
    # Классическая формула Симпсона
    s_odd = 0.0
    s_even = 0.0
    for i in range(1, n, 2):
        s_odd += y[i]
    for i in range(2, n, 2):
        s_even += y[i]
    simpson_approx = (h/3.0)*(y[0] + y[n] + 4.0*s_odd + 2.0*s_even)

    return (simpson_approx, real_calls)

In [35]:
def f1(x: float) -> float:
    if x == 0:
        return 0.5
    return math.sin(x/2)/(math.e**x-1)

# "Имитация" точного значения с помощью большого числа разбиений Симпсона.
# (Если хотите исключить численную оценку, уберите пример 1.)
def get_almost_exact_for_f1():
    big_calls = 10_000_001
    val, _ = simpson_by_calls(f1, -1.0, 1.0, big_calls)
    return val

exact_val_1 = get_almost_exact_for_f1()

print(exact_val_1)

1.0130392362326186


In [41]:
eps = sys.float_info.epsilon
print(eps)

2.220446049250313e-16


In [45]:
a1, b1 = -1.0, 1.0

for m in range(1, 10):
    for J in range(1, 10):
        diff_approx, diff_calls = integrate_by_diff_scheme(f1, a1, b1, J, m)
        diff_err = abs(diff_approx - exact_val_1)
        
        if diff_err <= eps:
            print(f"m = {m}; J = {J}; I = {diff_approx} +- {diff_err}")
            

m = 7; J = 6; I = 1.0130392362326184 +- 2.220446049250313e-16


# Задача 2.

Используя методику прямой сшивки при взятии сумм рядов, рассчитать постоянную $\gamma_E$ Эйлера-Маскерони с ошибкой в 13 знаке по формуле $\gamma_E = \lim_{n\to\infty} (1 + 1/2 + 1/3 + 1/4 + \ldots + 1/n - \ln n)$

$$
s_n = 1 + 1/2 + 1/3 + 1/4 + \ldots + 1/n - \ln n \\
a_n = s_n - s_{n-1} = 1/n + \ln \left(\frac{n-1}{n}\right)\\
\gamma_E = \sum_{j=1}^\infty a_j
$$

$$
\sum_{j=1}^{\infty} a_j = \int_{j_m + 1/2}^{\infty} a_j dj + \sum_{j=1}^{j_m - m} a_j + \sum_{j=j_m - m + 1}^{j_m + m} (1 - I^m_{j+m-j_m-1}) a_j + O(1/j_{\text{s}}^{2m+2})\\
I_l^m = \sum_{k=0}^{l} W_{2m-k}^m
$$

In [73]:
@lru_cache(None)
def I_list(w_list, m):
    result = []
    
    for l in range(2*m):
        result.append(sum(w_list[2*m-l:]))
    
    return result

In [136]:
def a_list(n):
    result = [0, 1]

    for i in range(2, n+1):
        result.append(1/i + math.log((i-1)/i))
        
    return result

$$
\int_{j_m + 1/2}^{\infty} a_j dj = \int_{j_m + 1/2}^{B} a_j dj + \int_{B}^{\infty} a_j dj
$$

$$
a_j = \sum_{n=2}^\infty (-1)^{n+1} \frac{1}{nj^n}\\
\left| \int_B^\infty a_j \, dj \, \right| = \left| \sum_{n=2}^\infty \int_B^\infty (-1)^{n+1} \frac{dj}{nj^n} \right| = \left| \sum_{n=2}^\infty \frac{(-1)^{n}}{n(n-1)B^{n-1}} \right| < \left|\, для \, B > 2, \, \frac{1}{n(n-1)B^{n-1}} >  \frac{1}{(n+1)nB^n} \, \right| < 
\sum_{n=1}^\infty \frac{1}{2n(2n-1)B^{2n-1}} <\\
< \frac{1}{B} \sum_{n=1}^\infty \frac{1}{2n(2n-1)} < \frac{1}{B} \sum_{n=1}^\infty \frac{1}{2n^2} = \frac{1}{2B} \sum_{n=1}^\infty \frac{1}{n^2} = \frac{\pi^2}{12B} < \frac{1}{B} < \varepsilon\\
B > \frac{1}{\varepsilon}
$$

#### Но, можно вычислить интеграл аналитически.

$$
\int_{j_m + 1/2}^{\infty} a_j dj = \ln j + (j-1)\ln (j-1) - (j-1) - j \ln j + j \; \Big|_{j_m + 1/2}^\infty= 1 + (j-1)\ln\frac{j-1}{j} \; \Big|_{j_m + 1/2}^\infty = (j_m - 1/2) \ln \frac{j_m + 1/2}{j_m - 1/2} -1
$$

$$ j_s = 10; \; m = 7 \quad \text{даст необходимую точность} \; O(1/j_{\text{s}}^{2m+2})$$

In [143]:
a_list(100)[::10]

[0,
 -0.005360515657826276,
 -0.0012932943875505754,
 -0.0005682183423480064,
 -0.00031780798428989593,
 -0.00020270731751946547,
 -0.00014045164971462215,
 -0.00010302316638527084,
 -7.878220686007194e-05,
 -6.218948701407732e-05,
 -5.033585350145038e-05]

При $j > 30, |a_{j+10}|$ более чем на поядок меньше чем $|a_j|$, поэтому выберем $j_m = 30$

In [125]:
def sum_a(j_m, m):
    integ = (j_m-0.5)*math.log((j_m+0.5)/(j_m-0.5)) - 1

    a_vals = a_list(j_m+m)
    I_vals = I_list(tuple(w_vals), m)

    sum1 = sum(a_vals[1:j_m-m+1])

    sum2 = 0

    for j in range(j_m-m+1, j_m+m+1):
        sum2 += (1-I_vals[j+m-j_m-1])*a_vals[j]

    return integ + sum1 + sum2

In [202]:
m = 7
j_m = 30

print(sum_a(j_m, m))

0.5772156649015302


# Задача 3.

Вычислить сумму ряда

$$\sum_{n=1}^{\infty} \frac{n^2 + 1}{n^4 + n^2 + 1} \cos(2n)$$

с точностью $\varepsilon = 10^{-6}$, применяя метод Куммера.

$$
\sum_{n=1}^{\infty} \frac{\cos(2n)}{n^2} = \frac{\pi^2}{6} - \pi + 1 \\
\gamma = \lim_{n\to\infty} \frac{a_n}{b_n} = \lim_{n\to\infty} \frac{(n^2+1)\cos 2n \cdot n^2}{(n^4+n^2+1) \cdot \cos 2n} = 1\\
S =  \frac{\pi^2}{6} - \pi + 1 + \sum_{n=1}^{\infty} \left( \frac{n^2 + 1}{n^4+n^2+1} - \frac{1}{n^2} \right) \cdot \cos 2n = \frac{\pi^2}{6} - \pi + 1 - \sum_{n=1}^{\infty} \frac{\cos 2n}{n^2 (n^4 + n^2 + 1)}
$$

$$
|R_k| = \left| \sum_{n=k+1}^{\infty} \frac{\cos 2n}{n^2(n^4+n^2+1)} \right| \leq \sum_{n=k+1}^{\infty} \frac{1}{n^2(n^4+n^2+1)} < \sum_{n=k+1}^{\infty} \frac{1}{n^6} < \int_{k+1}^{\infty} \frac{dx}{x^6} = \frac{1}{5(k+1)^5} < 10^{-6} \\
k > 10
$$

In [191]:
def s_n(n):
    return math.cos(2*n)/(n**2*(n**4+n**2+1))

In [192]:
S = math.pi*(math.pi/6-1) + 1

for i in range(1, 12):
    S -= s_n(i)
    
print(S)

-0.3512652041632815


# Задача 4.

Взять предел

$$\lim_{x\to0} \frac{y(x) - 1}{x}$$

где

$$
y(x) = \begin{cases}
    \operatorname{ch}\sqrt{|x|}, & x > 0 \\
    \cos\sqrt{|x|}, & x < 0
\end{cases}
$$

Поиграться с выбором шага.

$$
f(x) =\frac{y(x) - 1}{x}\\
\lim_{x\to0} f(x) = \lim_{x\to0} \frac{f(x)+f(-x)}{2} = \frac{1}{2}\lim_{x\to0} \frac{y(x) - 1}{x} - \frac{y(-x) - 1}{x} = \lim_{x\to0} \frac{y(x)-y(-x)}{2x} =  \lim_{x\to0} \frac{ch\sqrt{|x|} - cos\sqrt{|x|}}{2|x|} = \lim_{x\to0} g(x)
$$

In [242]:
def g(x):
    return 0.5*(math.cosh(abs(x)**0.5)-math.cos(abs(x)**0.5))/abs(x)

In [243]:
def get_y(m, dx):
    y = []

    for i in range(2*m+2):
        y.append(g((2*i-1)*dx/2))
    return y

In [244]:
def lim_f(m, dx):
    y = get_y(m, dx)
    
    result = 0

    for n in range(2*m+2):
        for k in range(2*m+n%2):
            idx = 2*m + n%2 - 2*k
            if idx < 0:
                idx = -1*idx - 1
                
            a_val = float(A(k, n, m-int((n-n%2)/2)))
            fact = float(factorial_dec(n))
            
            result += a_val*(-1)**n/(4**n*fact)*y[idx]
            
    return result


In [266]:
def calculate_lim(m, dx):
    print(lim_f(m, dx))
    
interact(calculate_lim, m=widgets.IntSlider(min=1, max=10, step=1, value=5), \
         dx=widgets.FloatSlider(min=0.1, max=100, step=0.1, value=10))

interactive(children=(IntSlider(value=5, description='m', max=10, min=1), FloatSlider(value=10.0, description=…

<function __main__.calculate_lim(m, dx)>